# 11 — Interactive Visualization (yFiles)

This notebook demonstrates the `holonic.viz` module: interactive Jupyter widgets for visualizing holons, holarchies, and provenance trails.

Unlike notebook 05 (which builds projection data structures), this notebook renders actual graphs in the browser using `yfiles-jupyter-graphs`.

## Requirements

This notebook requires the optional `viz` extra, which pulls in `yfiles-jupyter-graphs` and `ipywidgets`. Run the install cell below if you haven't already.

In [ ]:
# Install optional visualization dependencies.
# Safe to re-run; pip will no-op if already installed.
%pip install --quiet "holonic[viz]"

## Setup — A Research Lab Holarchy

A compact three-tier holarchy: a research organization, two department holons, and a few team holons inside each department. Enough structure to produce a readable topology without overwhelming the visualization.

In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()

# Root organization
ds.add_holon("urn:holon:atlas-labs", "Atlas Labs")
ds.add_interior("urn:holon:atlas-labs", '''
    @prefix schema: <https://schema.org/> .
    <urn:org:atlas> a schema:Organization ;
        schema:name "Atlas Labs" ;
        schema:description "Applied AI and knowledge engineering." ;
        schema:numberOfEmployees 42 .
''')

# Two departments under the organization
for dept_iri, dept_label in [
    ("urn:holon:research", "Research"),
    ("urn:holon:engineering", "Engineering"),
]:
    ds.add_holon(dept_iri, dept_label, member_of="urn:holon:atlas-labs")
    ds.add_interior(dept_iri, f'''
        @prefix schema: <https://schema.org/> .
        <{dept_iri.replace("holon:", "dept:")}> a schema:Organization ;
            schema:name "{dept_label}" ;
            schema:parentOrganization <urn:org:atlas> .
    ''')
    ds.add_boundary(dept_iri, '''
        @prefix schema: <https://schema.org/> .
        @prefix sh: <http://www.w3.org/ns/shacl#> .
        [] a sh:NodeShape ;
            sh:targetClass schema:Organization ;
            sh:property [
                sh:path schema:name ;
                sh:minCount 1 ;
                sh:maxCount 1 ;
                sh:datatype xsd:string
            ] .
    ''')

# Teams inside each department
teams = [
    ("urn:holon:kg-team", "Knowledge Graphs Team", "urn:holon:research"),
    ("urn:holon:ml-team", "Machine Learning Team", "urn:holon:research"),
    ("urn:holon:platform-team", "Platform Team", "urn:holon:engineering"),
    ("urn:holon:tools-team", "Developer Tools Team", "urn:holon:engineering"),
]

for team_iri, team_label, parent in teams:
    ds.add_holon(team_iri, team_label, member_of=parent)
    ds.add_interior(team_iri, f'''
        @prefix schema: <https://schema.org/> .
        <{team_iri.replace("holon:", "team:")}> a schema:Organization ;
            schema:name "{team_label}" ;
            schema:memberOf <{parent.replace("holon:", "dept:")}> .
    ''')

# A cross-department portal: research tools shared with engineering
ds.add_portal(
    "urn:portal:kg-to-platform",
    source_iri="urn:holon:kg-team",
    target_iri="urn:holon:platform-team",
    construct_query='''
        PREFIX schema: <https://schema.org/>
        CONSTRUCT { ?x schema:sameAs ?x }
        WHERE { GRAPH ?g { ?x a schema:Organization } }
    ''',
    label="KG team → Platform team",
)

print(f"Holons: {len(ds.list_holons())}")
print(f"Portals: {len(ds.list_portals())}")

## Holarchy Topology (collapsed view)

`HolarchyViz` shows the holons and their containment relationships as a graph. The collapsed view treats each holon as a single node — interior, boundary, projection, and context graphs are hidden.

In [ ]:
from holonic.viz import HolarchyViz

hz = HolarchyViz(ds, layout="hierarchic")
hz.show()

## Holarchy Topology (expanded — interior and boundary layers visible)

The expanded view shows each holon as a cluster including its layer graphs. Useful for understanding how holon boundaries and interiors relate to the topology.

In [ ]:
hz_expanded = HolarchyViz(
    ds,
    show_internals=True,
    layers=["interior", "boundary"],
    layout="hierarchic",
)
hz_expanded.show()

## Single Holon View

`HolonViz` focuses on one holon and renders its interior triples, boundary shapes, and (if present) context graph. SHACL shapes are rendered as compartmented tables instead of blank-node trees.

In [ ]:
from holonic.viz import HolonViz

hv = HolonViz(
    ds,
    "urn:holon:kg-team",
    layers=["interior", "boundary"],
)
hv.show()

## SPARQL Explorer

`SPARQLExplorer` lets you run CONSTRUCT queries against the dataset and render the resulting graph inline. It comes with a set of built-in projection presets (holarchy structure, shape constraints, provenance trails) and supports custom SPARQL.

Running the explorer below displays a panel where you can select presets or edit queries directly.

In [ ]:
from holonic.viz import SPARQLExplorer

explorer = SPARQLExplorer(ds)
explorer.show()

## Provenance Trail

Run a couple of traversals to generate provenance records, then visualize them with `ProvenanceViz`. Each traversal produces a `prov:Activity` in the target holon's context graph; the visualization shows the chain of activities with agent attribution.

In [ ]:
# Fire a traversal to generate a provenance record
projected, membrane = ds.traverse(
    source_iri="urn:holon:kg-team",
    target_iri="urn:holon:platform-team",
    validate=False,  # CONSTRUCT produces data that doesn't match the boundary shape
    agent_iri="urn:agent:research-engineer",
)

print(f"Generated {len(projected)} triples, membrane health = {membrane.health.value if membrane else 'not validated'}")

In [ ]:
from holonic.viz import ProvenanceViz

pv = ProvenanceViz(ds, show_surface=False)
pv.show()

In [ ]:
# Textual provenance summary (useful when running headless)
pv.print_report()

## Where to Go Next

The `holonic.viz` module also provides lower-level builders for custom workflows:

- `holon_to_yfiles(ds, iri, ...)` — raw yFiles node/edge data for a single holon
- `holarchy_to_yfiles(ds, ...)` — raw node/edge data for the holarchy topology
- `projected_to_yfiles(projected_graph)` — convert any `ProjectedGraph` to yFiles format
- `sparql_result_to_yfiles(result_graph, ...)` — render a CONSTRUCT result directly

For styling, see `holonic.viz.styles` for the default color / shape palette. For label formatting, `format_compartmented`, `format_typed`, `format_shacl_shape`, and `format_simple` in `holonic.viz.formatters` cover the common cases.

Notebook 05 (`05_holarchy_viz.ipynb`) covers the data-structure side of holarchy visualization — projection to LPG, graphology export for sigma.js, NetworkX integration — for callers who don't want the yFiles dependency.